In [16]:
import json
import math
import random
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional


import numpy as np
import pandas as pd

In [17]:
from pathlib import Path
import json
import random
import numpy as np

BASE_DIR = Path.cwd()
ASSIGNED_POST = BASE_DIR / "outputs_json" / "artwork_assigned_v2_post.json"
VEC_PATH = Path.cwd() / "outputs_json" / "artwork_vector.json"  # 또는 post 파일
vec_raw = json.load(open(VEC_PATH, "r", encoding="utf-8"))

def build_vec_dict(vec_raw):
    # dict 포맷: { "category000_0000.png": [vec], ... }
    if isinstance(vec_raw, dict):
        return {Path(str(k)).stem: v for k, v in vec_raw.items()}

    # list 포맷: [{"artwork_id":"...","artwork_vector":[...]}, ...]
    out = {}
    for r in vec_raw:
        if not isinstance(r, dict): 
            continue
        vid = r.get("artwork_id") or r.get("item_id") or r.get("id")
        vec = r.get("artwork_vector") or r.get("vector") or r.get("embedding")
        if vid is None or vec is None:
            continue
        out[Path(str(vid)).stem] = vec
    return out

artwork_vector = build_vec_dict(vec_raw)
AVAILABLE_IDS = list(artwork_vector.keys())

print("vector keys:", len(AVAILABLE_IDS), "sample:", AVAILABLE_IDS[:10])

def load_json(p: Path):
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)

from pathlib import Path

def parse_vector_json(vec_data):
    """
    returns vec_dict: {artwork_id_stem(str): vector(list[float])}
    supports:
      - dict: {id: [vec]}
      - list: [{"artwork_id":..., "artwork_vector":[...]}] (or item_id/vector/embedding)
    """
    vec_dict = {}

    def _stem(x):
        return Path(str(x)).stem  # '.../a.png' -> 'a'

    if isinstance(vec_data, dict):
        for k, v in vec_data.items():
            if isinstance(v, list):
                vec_dict[_stem(k)] = v
        return vec_dict

    if isinstance(vec_data, list):
        for r in vec_data:
            if not isinstance(r, dict):
                continue
            vid = r.get("artwork_id") or r.get("item_id") or r.get("id")
            vec = r.get("artwork_vector") or r.get("vector") or r.get("embedding")
            if vid is None or vec is None:
                continue
            vec_dict[_stem(vid)] = vec
        return vec_dict

    raise ValueError("Unsupported artwork_vector.json format")

vec_raw  = load_json(VEC_PATH)
vec_dict = parse_vector_json(vec_raw)   # ✅ 이제 stem 키로 통일됨

assigned = load_json(ASSIGNED_POST)

assigned_ids = [str(r["artwork_id"]) for r in assigned if isinstance(r, dict) and r.get("artwork_id") is not None]
assigned_set = set(assigned_ids)
vec_set = set(vec_dict.keys())

inter = assigned_set & vec_set
missing = list(assigned_set - vec_set)

print("assigned unique:", len(assigned_set))
print("vector unique  :", len(vec_set))
print("matched        :", len(inter))
print("missing        :", len(missing))
print("match rate     :", (len(inter) / max(len(assigned_set), 1)) * 100, "%")

# 벡터 차원 체크 (샘플 10개)
sample_ids = random.sample(list(inter), k=min(10, len(inter))) if inter else []
dims = []
bad = []
for aid in sample_ids:
    v = vec_dict[aid]
    if not isinstance(v, list):
        bad.append(aid)
        continue
    dims.append(len(v))

print("\nvector dim sample:", dims[:10])
if dims:
    print("dim min/max:", min(dims), max(dims))
if bad:
    print("non-list vector sample ids:", bad[:5])

# 누락 샘플 출력
print("\nmissing sample:", missing[:20])

# backward-compatible alias
AVAILABLE_ARTWORK_IDS = AVAILABLE_IDS


vector keys: 5000 sample: ['category030_0006', 'category030_0030', 'category030_0017', 'category030_0028', 'category030_0002', 'category030_0035', 'category030_0027', 'category030_0025', 'category030_0023', 'category030_0033']
assigned unique: 5000
vector unique  : 5000
matched        : 5000
missing        : 0
match rate     : 100.0 %

vector dim sample: [512, 512, 512, 512, 512, 512, 512, 512, 512, 512]
dim min/max: 512 512

missing sample: []


In [18]:
# =========================
# Config (여기만 바꾸면 됨)
# =========================


DATA_DIR = Path("./outputs_json")


ARTWORK_PATH = DATA_DIR / "artwork_assigned_v2_post.json"
ARTIST_SUMMARY_PATH = DATA_DIR / "artists_summary_v2.json"
ARTWORK_VECTOR_PATH = DATA_DIR / "artwork_vector.json"

SEED = 42


# 유저 수
N_USERS = 20000


# 유저 타입 비율 (합=1 권장)
COLD_RATIO = 0.10
NORMAL_RATIO = 0.50
HEAVY_RATIO = 0.40


# 타입별 로그 길이 범위 (min,max)  ※ "기준"은 고정, 유저별 길이는 이 범위에서 샘플
COLD_LEN_RANGE = (1, 4)
NORMAL_LEN_RANGE = (5, 30)
HEAVY_LEN_RANGE = (30, 200)


# 임베딩 벡터 차원
VECTOR_DIM = 512


# 벡터 합성 강도 (학습이 "쉽게" 되도록 카테고리/작가 패턴을 강하게 넣는 파라미터)
W_CATEGORY = 0.65
W_ARTIST = 0.30
W_NOISE = 0.05  # 너무 크면 패턴이 약해짐


# 시간 생성 (최신순 정렬용)
# 한 유저의 이벤트들이 이 기간 안에서 발생한 것으로 생성됨
USER_ACTIVITY_DAYS = 60


# action_type 종류
ACTIONS = ["VIEW", "LIKE", "STAY", "REVIEW", "COMMENT"]


# action_type 기본 확률 (선호도/반복조회 등에 의해 가중될 것)
BASE_ACTION_PROBS = {
    "VIEW": 0.72,
    "STAY": 0.18,     # 10초 이상 체류
    "LIKE": 0.08,
    "REVIEW": 0.015,
    "COMMENT": 0.005,
}


# 출력 경로
OUT_DIR = DATA_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)


OUT_JSONL = OUT_DIR / "user_logs_detail.jsonl"
OUT_PARQUET = OUT_DIR / "user_logs_detail.parquet"
OUT_INPUT3_JSONL = OUT_DIR / "user_logs.jsonl"


print("OUT_DIR:", OUT_DIR)

OUT_DIR: outputs_json


In [19]:
def set_all_seeds(seed: int):
    random.seed(seed)
    np.random.seed(seed)

set_all_seeds(SEED)

def read_json(path: Path) -> Any:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def softmax(x: np.ndarray, temp: float = 1.0) -> np.ndarray:
    x = x / max(temp, 1e-8)
    x = x - np.max(x)
    e = np.exp(x)
    return e / np.sum(e)

def weighted_choice(items: List[Any], weights: List[float]) -> Any:
    # weights can be unnormalized
    return random.choices(items, weights=weights, k=1)[0]

def l2_normalize(v: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    n = np.linalg.norm(v)
    return v / (n + eps)

def now_utc():
    return datetime.now(timezone.utc)

def sample_int_in_range(rng: Tuple[int, int]) -> int:
    lo, hi = rng
    if lo == hi:
        return lo
    return random.randint(lo, hi)

In [20]:
artworks = read_json(ARTWORK_PATH)
artist_summaries = read_json(ARTIST_SUMMARY_PATH)

print("Loaded artworks:", len(artworks))
print("Loaded artist summaries:", len(artist_summaries))

# artwork 구조에서 필요한 필드만 추출/정리
# artwork_assigned_v2.json 예시 필드:
# artwork_id, artist_id, primary_genre, topk_genres, genre(list), top1_cluster ...
artwork_rows = []
for a in artworks:
    artwork_rows.append({
        "artwork_id": Path(str(a.get("output_path") or a.get("artwork_id"))).stem,
        "artwork_id_raw": a.get("artwork_id"),
        "artist_id": a.get("artist_id"),
        "primary_genre": a.get("primary_genre"),
        "genres": a.get("genre") or a.get("topk_genres") or ([a.get("primary_genre")] if a.get("primary_genre") else []),
        "top1_cluster": a.get("top1_cluster"),
    })

df_art = pd.DataFrame(artwork_rows)
df_art = df_art.dropna(subset=["artwork_id", "artist_id", "primary_genre"])
df_art["genres"] = df_art["genres"].apply(lambda x: x if isinstance(x, list) else [x])

print(df_art.head(3))
print("Unique artists:", df_art["artist_id"].nunique())
print("Unique primary_genre:", df_art["primary_genre"].nunique())

Loaded artworks: 5000
Loaded artist summaries: 750
         artwork_id    artwork_id_raw      artist_id primary_genre  \
0  category059_0001  category059_0001  v_artist_0000   category059   
1  category100_0001  category100_0001  v_artist_0001   category100   
2  category059_0002  category059_0002  v_artist_0002   category059   

                                              genres  top1_cluster  
0  [category059, category049, category070, catego...            11  
1  [category100, category094, category106, catego...            16  
2                         [category059, category070]            11  
Unique artists: 750
Unique primary_genre: 93


In [21]:
from collections import defaultdict

# category co-occurrence graph
co = defaultdict(lambda: defaultdict(int))

for _, row in df_art.iterrows():
    gs = list(dict.fromkeys(row["genres"]))  # unique preserving order
    for i in range(len(gs)):
        for j in range(i+1, len(gs)):
            co[gs[i]][gs[j]] += 1
            co[gs[j]][gs[i]] += 1

all_categories = sorted(set(df_art["primary_genre"].tolist()) | set(co.keys()))
cat2idx = {c:i for i,c in enumerate(all_categories)}

# neighbor list for each category (상위 co-occurrence K개)
K_NEIGHBOR = 10
cat_neighbors: Dict[str, List[str]] = {}

for c in all_categories:
    neigh = co.get(c, {})
    if len(neigh) == 0:
        cat_neighbors[c] = []
        continue
    sorted_neigh = sorted(neigh.items(), key=lambda x: x[1], reverse=True)
    cat_neighbors[c] = [n for n,_ in sorted_neigh[:K_NEIGHBOR]]

print("Example neighbors:")
for c in all_categories[:5]:
    print(c, "->", cat_neighbors[c][:5])

Example neighbors:
category001 -> ['category070', 'category049', 'category073', 'category084', 'category017']
category002 -> ['category094', 'category091', 'category101', 'category100', 'category077']
category003 -> ['category038', 'category051', 'category065', 'category057', 'category096']
category004 -> ['category067', 'category101', 'category097', 'category094', 'category100']
category005 -> ['category094', 'category101', 'category026', 'category095', 'category097']


In [22]:
# Cell 6 (FIXED): artwork vectors + meta (중복 없이 1:1 매핑)

# category prototype vectors
rng = np.random.default_rng(SEED)
cat_proto = {c: l2_normalize(rng.normal(size=(VECTOR_DIM,)).astype(np.float32)) for c in all_categories}

# artist prototype vectors: 평균 카테고리(작가 작품 primary_genre 기반)
artist_ids = sorted(df_art["artist_id"].unique().tolist())
artist_to_cats = df_art.groupby("artist_id")["primary_genre"].apply(list).to_dict()

artist_proto = {}
for aid in artist_ids:
    cats = artist_to_cats.get(aid, [])
    if not cats:
        artist_proto[aid] = l2_normalize(rng.normal(size=(VECTOR_DIM,)).astype(np.float32))
        continue
    vec = np.mean([cat_proto[c] for c in cats], axis=0)
    artist_proto[aid] = l2_normalize(vec.astype(np.float32))

# ✅ 핵심: AVAILABLE_IDS를 셔플해서 df_art 행과 1:1로 붙임(중복 덮어쓰기 방지)
ids_shuffled = AVAILABLE_IDS.copy()
random.shuffle(ids_shuffled)

n_need = len(df_art)
if len(ids_shuffled) < n_need:
    # 부족하면 반복(보통 5,000 vs 5,000이면 여기 안 탐)
    mul = (n_need // len(ids_shuffled)) + 1
    ids_shuffled = (ids_shuffled * mul)[:n_need]
else:
    ids_shuffled = ids_shuffled[:n_need]

# artwork vectors
artwork_vector: Dict[str, np.ndarray] = {}
artwork_meta: Dict[str, Dict[str, Any]] = {}

for wid, (_, row) in zip(ids_shuffled, df_art.iterrows()):
    aid = row["artist_id"]
    primary = row["primary_genre"]

    v = (
        W_CATEGORY * cat_proto[primary]
        + W_ARTIST * artist_proto[aid]
        + W_NOISE * l2_normalize(rng.normal(size=(VECTOR_DIM,)).astype(np.float32))
    )
    v = l2_normalize(v.astype(np.float32))

    artwork_vector[wid] = v
    artwork_meta[wid] = {
        "artist_id": aid,
        "primary_genre": primary,
        "top1_cluster": row.get("top1_cluster"),
        "genres": row["genres"],
    }

print("Artwork vectors:", len(artwork_vector))
print("Sample wid:", list(artwork_vector.keys())[:3])
print("Sample vector dim:", next(iter(artwork_vector.values())).shape)

Artwork vectors: 5000
Sample wid: ['category060_0095', 'category100_0071', 'category096_0535']
Sample vector dim: (512,)


In [23]:
@dataclass
class UserProfile:
    member_id: str
    user_type: str  # "cold"|"normal"|"heavy"
    fav_categories: List[str]
    fav_weights: List[float]
    exploration: float  # 0~1 (높을수록 랜덤 탐색)
    drift: float        # 0~1 (헤비 유저용, 높을수록 취향 변화)

def make_user_id(i: int) -> str:
    return f"user_{i:05d}"

def sample_fav_categories(user_type: str) -> Tuple[List[str], List[float], float, float]:
    if user_type == "cold":
        k = random.randint(1, 2)
        exploration = 0.55
        drift = 0.0
    elif user_type == "normal":
        k = random.randint(3, 5)
        exploration = 0.35
        drift = 0.15
    else:  # heavy
        k = random.randint(5, 10)
        exploration = 0.20
        drift = 0.35

    fav = random.sample(all_categories, k=k)

    # 더 “학습 쉽게” 하려면 선호 가중치가 명확해야 해서 softmax로 뾰족하게 만듦
    raw = np.array(rng.normal(loc=0.0, scale=1.0, size=(k,)), dtype=np.float32)
    w = softmax(raw, temp=0.7).tolist()
    return fav, w, exploration, drift

def build_user_profiles(n_users: int) -> List[UserProfile]:
    n_cold = int(round(n_users * COLD_RATIO))
    n_normal = int(round(n_users * NORMAL_RATIO))
    n_heavy = n_users - n_cold - n_normal

    types = (["cold"] * n_cold) + (["normal"] * n_normal) + (["heavy"] * n_heavy)
    random.shuffle(types)

    profiles = []
    for i, t in enumerate(types):
        fav, w, exploration, drift = sample_fav_categories(t)
        profiles.append(UserProfile(
            member_id=make_user_id(i),
            user_type=t,
            fav_categories=fav,
            fav_weights=w,
            exploration=exploration,
            drift=drift
        ))
    return profiles

profiles = build_user_profiles(N_USERS)

pd.Series([p.user_type for p in profiles]).value_counts()

normal    10000
heavy      8000
cold       2000
Name: count, dtype: int64

In [29]:
# NEW CELL: choose_category_for_user / choose_artwork_by_category 정의

from collections import defaultdict

# 카테고리 -> 작품 목록
cat_to_wids = defaultdict(list)
for wid, meta in artwork_meta.items():
    cat_to_wids[meta["primary_genre"]].append(wid)

# fallback용 전체 목록
ALL_WIDS = list(artwork_meta.keys())

def choose_category_for_user(p: UserProfile, step: int, total_steps: int) -> str:
    """
    유저 선호 카테고리 기반 + 탐색(exploration) + 드리프트(drift)
    - exploration: 랜덤/이웃 카테고리로 튀는 비율
    - drift: heavy 유저일수록 뒤로 갈수록 취향이 조금 이동
    """
    # 1) 기본: 선호 카테고리에서 weighted pick
    base_cat = random.choices(p.fav_categories, weights=p.fav_weights, k=1)[0]

    # 2) 드리프트(heavy): 후반부일수록 이웃으로 이동 확률 증가
    drift_prob = p.drift * (step / max(total_steps - 1, 1))

    # 3) exploration: 일정 확률로 랜덤/이웃 선택
    r = random.random()
    if r < p.exploration:
        # exploration 안에서는 절반은 이웃, 절반은 완전 랜덤
        if random.random() < 0.6 and cat_neighbors.get(base_cat):
            return random.choice(cat_neighbors[base_cat])
        return random.choice(all_categories)

    # drift 구간
    if random.random() < drift_prob and cat_neighbors.get(base_cat):
        return random.choice(cat_neighbors[base_cat])

    return base_cat

def choose_artwork_by_category(cat: str) -> str:
    """카테고리 안에서 랜덤 작품 선택 (없으면 전체에서 fallback)"""
    pool = cat_to_wids.get(cat, [])
    if pool:
        return random.choice(pool)
    return random.choice(ALL_WIDS)

In [30]:
def sample_num_events(user_type: str) -> int:
    if user_type == "cold":
        return random.randint(*COLD_LEN_RANGE)
    if user_type == "normal":
        return random.randint(*NORMAL_LEN_RANGE)
    return random.randint(*HEAVY_LEN_RANGE)

def generate_user_events(p: UserProfile):
    events = []
    seen = {}

    total_steps = sample_num_events(p.user_type)

    # 시간 범위: 최근 USER_ACTIVITY_DAYS일 안에서 랜덤 발생
    now = datetime.now(timezone.utc)
    start = now - timedelta(days=USER_ACTIVITY_DAYS)
    span = (now - start).total_seconds()

    for step in range(total_steps):
        cat = choose_category_for_user(p, step=step, total_steps=total_steps)
        wid = choose_artwork_by_category(cat)  # categoryXXX_XXXX 형태로 나와야 함

        repeat_view = wid in seen
        seen[wid] = seen.get(wid, 0) + 1

        action = sample_action(p, cat=cat, repeat_view=repeat_view)

        t = start + timedelta(seconds=random.random() * span)

        events.append({
            "member_id": p.member_id,
            "artwork_id": wid,
            "action_type": action,
            "event_time": t.isoformat(),
        })

    # 최신순 정렬: 1줄이 가장 최신
    events.sort(key=lambda x: x["event_time"], reverse=True)
    return events

print("OK!")

OK!


In [32]:
# Cell: FINAL generate_user_events (NO sample_action)

from datetime import datetime, timedelta, timezone
import random
import numpy as np
import uuid

# --- follow-up(뷰 이후) 액션 확률: VIEW는 항상 생성이므로 제외 ---
FOLLOWUP_BASE = {
    "STAY":    BASE_ACTION_PROBS.get("STAY", 0.18),
    "LIKE":    BASE_ACTION_PROBS.get("LIKE", 0.08),
    "COMMENT": BASE_ACTION_PROBS.get("COMMENT", 0.005),
    "REVIEW":  BASE_ACTION_PROBS.get("REVIEW", 0.015),
}

def clamp01(x: float) -> float:
    return max(0.0, min(1.0, float(x)))

def followup_probs(p: UserProfile, cat: str, repeat_view: bool):
    probs = dict(FOLLOWUP_BASE)

    if cat in p.fav_categories:
        probs["LIKE"] *= 1.8
        probs["STAY"] *= 1.3
        probs["REVIEW"] *= 1.6
        probs["COMMENT"] *= 1.4

    if repeat_view:
        probs["STAY"] *= 1.25
        probs["LIKE"] *= 1.05
        probs["REVIEW"] *= 1.10
        probs["COMMENT"] *= 1.10

    if getattr(p, "user_type", "") == "cold":
        probs["LIKE"] *= 0.6
        probs["STAY"] *= 0.7
        probs["COMMENT"] *= 0.5
        probs["REVIEW"] *= 0.4
    elif getattr(p, "user_type", "") == "heavy":
        probs["LIKE"] *= 1.2
        probs["STAY"] *= 1.1
        probs["COMMENT"] *= 1.3
        probs["REVIEW"] *= 1.2

    for k in probs:
        probs[k] = clamp01(probs[k])

    return probs

def sample_followups(p: UserProfile, cat: str, repeat_view: bool):
    probs = followup_probs(p, cat, repeat_view)
    outs = []

    if random.random() < probs["STAY"]:
        stay_sec = int(np.random.randint(10, 301))
        outs.append(("STAY", {"stay_seconds": stay_sec}))

    if random.random() < probs["LIKE"]:
        outs.append(("LIKE", {"value": 1}))

    if random.random() < probs["COMMENT"]:
        outs.append(("COMMENT", {"text": "nice work"}))

    if random.random() < probs["REVIEW"]:
        rating = int(np.random.randint(1, 6))
        outs.append(("REVIEW", {"rating": rating, "text": "great"}))

    return outs

def sample_num_views(user_type: str) -> int:
    if user_type == "cold":
        return random.randint(*COLD_LEN_RANGE)
    if user_type == "normal":
        return random.randint(*NORMAL_LEN_RANGE)
    return random.randint(*HEAVY_LEN_RANGE)

def generate_user_events(p: UserProfile):
    events = []
    seen = {}

    total_views = sample_num_views(p.user_type)

    now = datetime.now(timezone.utc)
    start = now - timedelta(days=USER_ACTIVITY_DAYS)
    span = (now - start).total_seconds()

    base_offsets = {
        "LIKE": (2, 120),
        "STAY": (10, 400),
        "COMMENT": (30, 800),
        "REVIEW": (60, 1800),
    }

    for step in range(total_views):
        cat = choose_category_for_user(p, step=step, total_steps=total_views)
        wid = choose_artwork_by_category(cat)

        repeat_view = wid in seen
        seen[wid] = seen.get(wid, 0) + 1

        # ✅ 이 VIEW 묶음의 고유 ID
        view_event_id = f"{p.member_id}_{step}_{uuid.uuid4().hex[:8]}"

        t_view = start + timedelta(seconds=random.random() * span)

        # 1) VIEW
        events.append({
            "member_id": p.member_id,
            "artwork_id": wid,
            "action_type": "VIEW",
            "event_time": t_view.isoformat(),

            # ✅ 그룹 정보
            "view_event_id": view_event_id,
            "parent_action": None,
            "parent_view_time": None,
        })

        # 2) FOLLOW-UP (반드시 이 VIEW의 child)
        for act, extra in sample_followups(p, cat=cat, repeat_view=repeat_view):
            lo, hi = base_offsets.get(act, (5, 300))
            t2 = t_view + timedelta(seconds=int(np.random.randint(lo, hi + 1)))

            row = {
                "member_id": p.member_id,
                "artwork_id": wid,
                "action_type": act,
                "event_time": t2.isoformat(),

                # ✅ 반드시 VIEW에 종속됨
                "view_event_id": view_event_id,
                "parent_action": "VIEW",
                "parent_view_time": t_view.isoformat(),
            }
            if extra:
                row.update(extra)
            events.append(row)

    # 최신순 정렬은 유지해도 됨 (관계는 view_event_id로 보장)
    events.sort(key=lambda x: x["event_time"], reverse=True)
    return events

print("[OK] generate_user_events = VIEW + followups (NO sample_action)")

[OK] generate_user_events = VIEW + followups (NO sample_action)


In [35]:
# Cell 13 (FIXED + GUARD): Save + Export
import json
import pandas as pd
from tqdm.auto import tqdm

# ✅ GUARD: df_logs가 없으면 생성
if "df_logs" not in globals():
    print("[WARN] df_logs not found. Generating df_logs now...")
    all_events = []
    for p in tqdm(profiles, desc="Generating user events"):
        all_events.extend(generate_user_events(p))
    df_logs = pd.DataFrame(all_events)
    if "event_time" in df_logs.columns and "member_id" in df_logs.columns:
        df_logs = df_logs.sort_values(["member_id", "event_time"], ascending=[True, True]).reset_index(drop=True)
    print("[OK] df_logs generated:", df_logs.shape)

# ✅ 출력 경로도 없으면 정의
if "OUT_PARQUET" not in globals():
    OUT_PARQUET = OUT_DIR / "user_logs_detail.parquet"
if "OUT_JSONL" not in globals():
    OUT_JSONL = OUT_DIR / "user_logs_detail.jsonl"
if "OUT_INPUT3_JSONL" not in globals():
    OUT_INPUT3_JSONL = OUT_DIR / "user_logs_input3.jsonl"

# 1) parquet 저장
df_logs.to_parquet(OUT_PARQUET, index=False)

# 2) JSONL 저장
with open(OUT_JSONL, "w", encoding="utf-8") as f:
    for row in tqdm(df_logs.itertuples(index=False), total=len(df_logs), desc="Writing JSONL"):
        f.write(json.dumps(row._asdict(), ensure_ascii=False) + "\n")

OUT_JSON = OUT_DIR / "user_logs_detail.json"
# 유저별로 묶고(오름차순), 유저 내부는 시간 오름차순(과거→현재)
df_logs = df_logs.sort_values(["member_id", "event_time"], ascending=[True, True]).reset_index(drop=True)

print("Saved:")
print(" -", OUT_PARQUET)
print(" -", OUT_JSONL)
print(" -", OUT_JSON)

def export_model_input_3cols(df: pd.DataFrame, use_views_only: bool = True, with_timestamp: bool = True):
    x = df.copy()

    if use_views_only and "action_type" in x.columns:
        x = x[x["action_type"] == "VIEW"].copy()

    if with_timestamp:
        # ✅ timestamp에 시간을 넣지 않고 action_type을 넣는다
        out = x[["member_id", "artwork_id", "action_type"]].copy()
        out = out.rename(columns={"action_type": "timestamp"})
    else:
        out = x[["member_id", "artwork_id"]].copy()

    with open(OUT_INPUT3_JSONL, "w", encoding="utf-8") as f:
        for row in tqdm(out.itertuples(index=False), total=len(out), desc="Writing model input JSONL"):
            f.write(json.dumps(row._asdict(), ensure_ascii=False) + "\n")

    OUT_INPUT3_JSON = OUT_DIR / "user_logs.json"
    out.to_json(OUT_INPUT3_JSON, orient="records", force_ascii=False)

    print("Saved model input:")
    print(" -", OUT_INPUT3_JSONL)
    print(" -", OUT_INPUT3_JSON)
    return out

df_input3 = export_model_input_3cols(df_logs, use_views_only=True, with_timestamp=True)
df_input3.head(5)
df_logs.head(30)[["member_id","action_type","event_time","artwork_id"]]

Writing JSONL: 100%|██████████| 1576991/1576991 [00:11<00:00, 135043.68it/s]


Saved:
 - outputs_json/user_logs_detail.parquet
 - outputs_json/user_logs_detail.jsonl
 - /home/j-i14e107/Image_classification/outputs_json/user_logs_detail.json


Writing model input JSONL: 100%|██████████| 1102342/1102342 [00:04<00:00, 238946.91it/s]


Saved model input:
 - outputs_json/user_logs.jsonl
 - /home/j-i14e107/Image_classification/outputs_json/user_logs.json


,member_id,action_type,event_time,artwork_id
0,user_00000,VIEW,2025-11-27T10:14:57.939544+00:00,category100_0126
1,user_00000,STAY,2025-11-27T10:15:57.939544+00:00,category100_0126
2,user_00000,VIEW,2025-11-28T01:24:17.268533+00:00,category101_0123
3,user_00000,STAY,2025-11-28T01:27:34.268533+00:00,category101_0123
4,user_00000,VIEW,2025-11-28T04:30:31.201735+00:00,category098_0087
5,user_00000,LIKE,2025-11-28T04:30:36.201735+00:00,category098_0087
6,user_00000,VIEW,2025-11-29T07:38:11.374589+00:00,category009_0006
7,user_00000,VIEW,2025-11-29T13:35:21.542624+00:00,category099_0002
8,user_00000,VIEW,2025-11-29T15:17:40.063390+00:00,category056_0251
9,user_00000,VIEW,2025-11-29T16:40:50.488689+00:00,category094_0374


In [28]:
from pathlib import Path

def stem_id(x: str) -> str:
    return Path(str(x)).stem  # "category096_0001.png" or "/.../category096_0001.png" -> "category096_0001"

def load_user_sequences(log_path: str, artwork2idx: dict, logs_are_latest_first: bool = True):
    logs = read_json_or_jsonl(log_path)
    if not isinstance(logs, list):
        raise ValueError("Train log must be JSON array or JSONL list")

    user_seq = {}
    missed = 0

    for log in logs:
        if not isinstance(log, dict):
            continue

        uid = str(log.get("member_id"))
        aid = log.get("artwork_id")

        if aid is None:
            missed += 1
            continue

        aid = stem_id(aid)  # 핵심: 규칙 통일
        if aid in artwork2idx:
            user_seq.setdefault(uid, []).append(artwork2idx[aid])
        else:
            missed += 1

    # 로그가 최신순이면 SASRec 학습용(과거->현재)으로 뒤집기
    if logs_are_latest_first:
        for uid in user_seq:
            user_seq[uid] = list(reversed(user_seq[uid]))

    return user_seq, missed

print("OK!")

OK!
